# full model comparison -- 18-class e-waste
**SDG 12.4 | Predictive Analysis Project**  
10 models: traditional ML + deep learning + hierarchical pipeline  
depends on: 02_cnn_classification.ipynb (trained models required)

## imports

In [ ]:
import os, time, json, warnings, pickle, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, ConcatDataset
from torchvision import datasets, transforms, models
from torch.cuda.amp import autocast

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                              classification_report, confusion_matrix)
from tqdm import tqdm

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")


## configuration -- 18 classes

In [ ]:
from pathlib import Path
import torch.nn as nn

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# paths
DATA_DIR   = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "models" / "comparison"
GRAPHS_DIR = OUTPUT_DIR / "graphs"
MODELS_DIR = OUTPUT_DIR / "saved_models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GRAPHS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

CONFIG = {
    "img_size"       : 224,
    "batch_size"     : 32,
    "num_epochs"     : 30,
    "lr"             : 1e-4,
    "weight_decay"   : 1e-4,
    "num_workers"    : 4,
    "patience"       : 7,
    "unfreeze_epoch" : 5,
    "embed_batch"    : 64,
}

# 18 classes -- final structure after all merges
# high (8): refrigerant-containing appliances + high-toxicity devices
# medium (4): large appliances with recoverable materials
# low (6): passive components + plastic-dominant devices

HAZARD_MAP = {
    "Battery"           : "HIGH",     # lithium / cadmium -- Basel Annex I
    "PCB"               : "HIGH",     # lead solder / brominated flame retardants
    "Mobile"            : "HIGH",     # lithium + rare earth metals
    "Television"        : "HIGH",     # crt lead glass / mercury
    "Laptop"            : "HIGH",     # lithium + lead solder
    "light bulbs"       : "HIGH",     # mercury (CFL) -- Basel Annex I
    "Refrigerator"      : "HIGH",     # CFC / HCFC refrigerants -- ozone-depleting
    "Air-Conditioner"   : "HIGH",     # HCFC refrigerants -- Basel Convention
    "Microwave"         : "MEDIUM",   # magnetron / steel -- recoverable
    "Washing Machine"   : "MEDIUM",   # steel / copper motor -- recoverable
    "Printer"           : "MEDIUM",   # toner / circuit -- moderate hazard
    "Microchip-IC"      : "MEDIUM",   # silicon / gold traces -- recoverable
    "Keyboard"          : "LOW",      # ABS plastic -- recyclable
    "Mouse"             : "LOW",      # ABS plastic -- recyclable
    "Resistor"          : "LOW",      # ceramic / carbon -- inert
    "transistor"        : "LOW",      # silicon / germanium -- recoverable
    "heat-sink"         : "LOW",      # aluminium -- high value recovery
    "Passive-Component" : "LOW",      # Capacitor + LED + semiconductor-diode merged
}

DISPOSAL_MAP = {
    "Battery"           : "hazardous waste facility -- lithium/cadmium recovery",
    "PCB"               : "certified e-waste recycler -- gold/copper extraction",
    "Mobile"            : "certified e-waste recycler -- rare earth recovery",
    "Television"        : "hazardous waste facility -- crt lead/mercury handling",
    "Laptop"            : "certified e-waste recycler -- battery + rare earth",
    "light bulbs"       : "hazardous waste facility -- mercury containment",
    "Refrigerator"      : "certified refrigerant recovery facility -- CFC/HCFC extraction before dismantling",
    "Air-Conditioner"   : "certified refrigerant recovery facility -- HCFC extraction before dismantling",
    "Microwave"         : "metal recycler -- steel/copper/magnetron recovery",
    "Washing Machine"   : "metal recycler -- steel/motor/copper recovery",
    "Printer"           : "e-waste recycler -- toner/circuit recovery",
    "Microchip-IC"      : "e-waste recycler -- silicon/gold recovery",
    "Keyboard"          : "plastic recycler -- abs plastic stream",
    "Mouse"             : "plastic recycler -- abs plastic stream",
    "Resistor"          : "component recycler -- ceramic/carbon recovery",
    "transistor"        : "component recycler -- semiconductor recovery",
    "heat-sink"         : "metal recycler -- aluminium recovery",
    "Passive-Component" : "component recycler -- semiconductor and aluminium recovery",
}

MATERIAL_MAP = {
    "Battery"           : "lithium / cadmium / lead acid",
    "PCB"               : "fr4 composite / lead solder / gold / copper",
    "Mobile"            : "lithium / rare earth metals / glass",
    "Television"        : "lead glass / mercury / plastic composite",
    "Laptop"            : "lithium / rare earth / aluminium / lead solder",
    "light bulbs"       : "glass / mercury / argon / tungsten",
    "Refrigerator"      : "steel / CFC-R12 or HCFC-R22 refrigerant / polyurethane foam / copper",
    "Air-Conditioner"   : "aluminium / HCFC-R22 or HFC-R410A refrigerant / copper / steel",
    "Microwave"         : "steel / copper / magnetron / ceramic",
    "Washing Machine"   : "steel / copper motor / rubber / plastic",
    "Printer"           : "abs plastic / toner / copper circuit / lead",
    "Microchip-IC"      : "silicon / gold bond wires / lead solder / ceramic",
    "Keyboard"          : "abs plastic / rubber / copper traces",
    "Mouse"             : "abs plastic / optical sensor / copper",
    "Resistor"          : "carbon / ceramic / metal film",
    "transistor"        : "silicon / germanium / plastic / lead",
    "heat-sink"         : "aluminium / copper / thermal compound",
    "Passive-Component" : "silicon / aluminium / gallium nitride / ceramic",
}

HAZARD_COLORS = {"HIGH": "#d7191c", "MEDIUM": "#fdae61", "LOW": "#1a9641"}
HAZARD_INT    = {"HIGH": 0, "MEDIUM": 1, "LOW": 2}

print("configuration loaded -- 18 classes")
for h in ["HIGH", "MEDIUM", "LOW"]:
    cls_list = [c for c, v in HAZARD_MAP.items() if v == h]
    print(f"  {h:<8}: {len(cls_list)} classes -- {cls_list}")



## data loaders

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
eval_transform = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = datasets.ImageFolder(DATA_DIR / "train", transform=eval_transform)
val_ds   = datasets.ImageFolder(DATA_DIR / "val",   transform=eval_transform)
test_ds  = datasets.ImageFolder(DATA_DIR / "test",  transform=eval_transform)

CLASS_NAMES = train_ds.classes
NUM_CLASSES = len(CLASS_NAMES)
assert NUM_CLASSES == 18, f"expected 18, got {NUM_CLASSES}"

test_loader  = DataLoader(test_ds, batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=CONFIG["num_workers"],
                          pin_memory=True)
embed_loader = DataLoader(
    ConcatDataset([train_ds, val_ds, test_ds]),
    batch_size=CONFIG["embed_batch"], shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=True)

full_labels = np.array(
    [s[1] for s in train_ds.samples] +
    [s[1] for s in val_ds.samples]   +
    [s[1] for s in test_ds.samples])

y_train = np.array([s[1] for s in train_ds.samples])
y_val   = np.array([s[1] for s in val_ds.samples])
y_test  = np.array([s[1] for s in test_ds.samples])

print(f"18 classes confirmed: {CLASS_NAMES}")
print(f"test samples: {len(y_test)}")


## load trained deep learning models

In [ ]:
CLS_DIR = PROJECT_ROOT / "models" / "classification"

def build_model(arch, num_classes):
    if arch == "resnet18":
        m = models.resnet18(weights=None)
        in_dim = m.fc.in_features
        m.fc = nn.Sequential(nn.Linear(in_dim, 512), nn.BatchNorm1d(512),
            nn.ReLU(inplace=True), nn.Dropout(0.4), nn.Linear(512, 256),
            nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(256, num_classes))
    elif arch == "resnet50":
        m = models.resnet50(weights=None)
        in_dim = m.fc.in_features
        m.fc = nn.Sequential(nn.Linear(in_dim, 512), nn.BatchNorm1d(512),
            nn.ReLU(inplace=True), nn.Dropout(0.4), nn.Linear(512, 256),
            nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(256, num_classes))
    elif arch == "efficientnet_b0":
        m = models.efficientnet_b0(weights=None)
        in_dim = m.classifier[1].in_features
        m.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(in_dim, 256),
            nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(256, num_classes))
    elif arch == "vit_b16":
        m = models.vit_b_16(weights=None)
        in_dim = m.heads.head.in_features
        m.heads.head = nn.Sequential(nn.Linear(in_dim, 512), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(512, num_classes))
    m.load_state_dict(torch.load(CLS_DIR / arch / f"{arch}_best.pth",
                                  map_location=device))
    return m.to(device)


ARCHS     = ["resnet18", "resnet50", "efficientnet_b0", "vit_b16"]
dl_models = {}
for arch in ARCHS:
    dl_models[arch] = build_model(arch, NUM_CLASSES)
    print(f"  loaded: {arch}")


## extract embeddings for traditional ml

In [ ]:
def extract_embeddings(model, arch, loader):
    if arch in ["resnet18", "resnet50"]:
        extractor = nn.Sequential(*list(model.children())[:-1])
    elif arch == "efficientnet_b0":
        extractor = nn.Sequential(model.features, model.avgpool)
    elif arch == "vit_b16":
        class ViTFeat(nn.Module):
            def __init__(self, v):
                super().__init__()
                self.v = v
            def forward(self, x):
                x   = self.v._process_input(x)
                cls = self.v.class_token.expand(x.shape[0], -1, -1)
                x   = torch.cat([cls, x], dim=1)
                x   = self.v.encoder(x)
                return x[:, 0]
        extractor = ViTFeat(model)

    extractor.eval().to(device)
    embs = []
    with torch.no_grad():
        for imgs, _ in tqdm(loader, desc=f"embed {arch}"):
            imgs = imgs.to(device, non_blocking=True)
            with autocast():
                f = extractor(imgs).squeeze(-1).squeeze(-1)
            embs.append(f.cpu().float().numpy())
    return np.vstack(embs)


print("extracting resnet50 embeddings for traditional ml...")
embs = extract_embeddings(dl_models["resnet50"], "resnet50", embed_loader)
print(f"embedding shape: {embs.shape}")

n_train = len(train_ds)
n_val   = len(val_ds)
pca     = PCA(n_components=256, random_state=42)
scaler  = StandardScaler()

X_train_raw = embs[:n_train]
X_val_raw   = embs[n_train:n_train + n_val]
X_test_raw  = embs[n_train + n_val:]

X_trainval = scaler.fit_transform(pca.fit_transform(np.vstack([X_train_raw, X_val_raw])))
X_test     = scaler.transform(pca.transform(X_test_raw))
y_trainval = np.concatenate([y_train, y_val])

print(f"trainval: {X_trainval.shape} | test: {X_test.shape}")


## traditional ml -- train and evaluate

In [ ]:
ML_MODELS = {
    "knn_k5"             : KNeighborsClassifier(n_neighbors=5, metric="cosine", n_jobs=-1),
    "knn_k11"            : KNeighborsClassifier(n_neighbors=11, metric="cosine", n_jobs=-1),
    "svm_rbf"            : SVC(kernel="rbf", C=10.0, gamma="scale", probability=True, random_state=42),
    "svm_linear"         : SVC(kernel="linear", C=1.0, probability=True, random_state=42),
    "random_forest"      : RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42),
    "logistic_regression": LogisticRegression(max_iter=2000, C=1.0, solver="lbfgs",
                                               multi_class="multinomial", random_state=42, n_jobs=-1),
    "naive_bayes"        : GaussianNB(),
    "gradient_boosting"  : GradientBoostingClassifier(n_estimators=200, max_depth=5,
                                                        learning_rate=0.1, random_state=42),
}

ml_results = {}
print(f"{'model':<25} {'accuracy':>10} {'macro_f1':>10} {'time':>8}")
print("-" * 58)

for name, clf in ML_MODELS.items():
    t0 = time.time()
    clf.fit(X_trainval, y_trainval)
    preds   = clf.predict(X_test)
    elapsed = time.time() - t0
    acc     = accuracy_score(y_test, preds)
    mf1     = f1_score(y_test, preds, average="macro")
    ml_results[name] = {
        "model_type": "traditional_ml", "accuracy": round(acc, 4),
        "macro_f1": round(mf1, 4),
        "weighted_f1": round(f1_score(y_test, preds, average="weighted"), 4),
        "macro_precision": round(precision_score(y_test, preds, average="macro"), 4),
        "macro_recall": round(recall_score(y_test, preds, average="macro"), 4),
        "train_time_s": round(elapsed, 2),
        "preds": preds.tolist(), "labels": y_test.tolist(),
    }
    with open(MODELS_DIR / f"{name}.pkl", "wb") as f:
        pickle.dump(clf, f)
    print(f"  {name:<23} {acc:>10.4f} {mf1:>10.4f} {elapsed:>7.1f}s")

print("traditional ml complete")


## deep learning evaluation on test set

In [ ]:
dl_results = {}
criterion  = nn.CrossEntropyLoss()

def eval_dl(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device, non_blocking=True)
            with autocast():
                out = model(imgs)
            _, p = torch.max(out, 1)
            preds.extend(p.cpu().numpy())
            labels.extend(lbls.numpy())
    return preds, labels


print(f"{'model':<25} {'accuracy':>10} {'macro_f1':>10}")
print("-" * 48)

for arch in ARCHS:
    preds, labels = eval_dl(dl_models[arch], test_loader)
    acc = accuracy_score(labels, preds)
    mf1 = f1_score(labels, preds, average="macro")
    dl_results[arch] = {
        "model_type": "deep_learning", "accuracy": round(acc, 4),
        "macro_f1": round(mf1, 4),
        "weighted_f1": round(f1_score(labels, preds, average="weighted"), 4),
        "macro_precision": round(precision_score(labels, preds, average="macro"), 4),
        "macro_recall": round(recall_score(labels, preds, average="macro"), 4),
        "preds": preds, "labels": labels,
    }
    print(f"  {arch:<23} {acc:>10.4f} {mf1:>10.4f}")

torch.cuda.empty_cache()


## hierarchical two-stage pipeline -- novel contribution

In [ ]:
HAZARD_INT_MAP = {"HIGH": 0, "MEDIUM": 1, "LOW": 2}
HAZARD_INT_INV = {v: k for k, v in HAZARD_INT_MAP.items()}

hazard_trainval = np.array([HAZARD_INT_MAP[HAZARD_MAP[CLASS_NAMES[l]]] for l in y_trainval])
hazard_test     = np.array([HAZARD_INT_MAP[HAZARD_MAP[CLASS_NAMES[l]]] for l in y_test])

# stage 1: hazard level classifier
print("training stage 1 -- hazard level (HIGH / MEDIUM / LOW)...")
stage1 = SVC(kernel="rbf", C=10.0, gamma="scale", probability=True, random_state=42)
stage1.fit(X_trainval, hazard_trainval)
s1_preds = stage1.predict(X_test)
s1_acc   = accuracy_score(hazard_test, s1_preds)
print(f"  stage 1 accuracy: {s1_acc:.4f}")

# stage 2: group-specific classifiers
hazard_groups = defaultdict(list)
for idx, cls in enumerate(CLASS_NAMES):
    hazard_groups[HAZARD_MAP[cls]].append(idx)

print("\ntraining stage 2 -- per-group classifiers...")
stage2_clfs  = {}
stage2_lmaps = {}

for hazard, cls_idxs in hazard_groups.items():
    mask_tr  = np.isin(y_trainval, cls_idxs)
    mask_te  = np.isin(y_test, cls_idxs)
    if mask_tr.sum() < 10:
        continue

    uniq      = sorted(np.unique(y_trainval[mask_tr]))
    local_map = {orig: loc for loc, orig in enumerate(uniq)}
    y_local   = np.array([local_map[l] for l in y_trainval[mask_tr]])

    clf = SVC(kernel="rbf", C=10.0, gamma="scale", probability=True, random_state=42)
    clf.fit(X_trainval[mask_tr], y_local)

    stage2_clfs[hazard]  = clf
    stage2_lmaps[hazard] = {v: k for k, v in local_map.items()}

    y_local_te   = np.array([local_map.get(l, 0) for l in y_test[mask_te]])
    grp_acc      = accuracy_score(y_local_te, clf.predict(X_test[mask_te]))
    print(f"  {hazard:<8} ({len(cls_idxs)} classes): accuracy = {grp_acc:.4f}")

# end-to-end evaluation
print("\nend-to-end hierarchical pipeline evaluation...")
hier_preds = []
for i in range(len(X_test)):
    ph  = HAZARD_INT_INV[stage1.predict(X_test[i:i+1])[0]]
    clf = stage2_clfs.get(ph)
    lp  = clf.predict(X_test[i:i+1])[0] if clf else 0
    hier_preds.append(stage2_lmaps.get(ph, {}).get(lp, 0))

hier_preds = np.array(hier_preds)
h_acc = accuracy_score(y_test, hier_preds)
h_mf1 = f1_score(y_test, hier_preds, average="macro")
print(f"\nhierarchical pipeline:")
print(f"  stage 1 hazard accuracy : {s1_acc:.4f}")
print(f"  end-to-end accuracy     : {h_acc:.4f}")
print(f"  end-to-end macro f1     : {h_mf1:.4f}")

hier_result = {
    "model_type": "hierarchical", "accuracy": round(h_acc, 4),
    "macro_f1": round(h_mf1, 4),
    "weighted_f1": round(f1_score(y_test, hier_preds, average="weighted"), 4),
    "macro_precision": round(precision_score(y_test, hier_preds, average="macro"), 4),
    "macro_recall": round(recall_score(y_test, hier_preds, average="macro"), 4),
    "stage1_hazard_accuracy": round(s1_acc, 4),
    "preds": hier_preds.tolist(), "labels": y_test.tolist(),
}


## combined results table

In [ ]:
all_results = {**ml_results, **dl_results, "hierarchical_svm": hier_result}
sorted_res  = sorted(all_results.items(), key=lambda x: x[1]["accuracy"], reverse=True)

print(f"\n{'rank':<5} {'model':<28} {'type':<20} {'accuracy':>10} {'macro_f1':>10}")
print("=" * 80)
for rank, (name, r) in enumerate(sorted_res, 1):
    marker = "  <-- best" if rank == 1 else ""
    print(f"  {rank:<4} {name:<28} {r['model_type']:<20} "
          f"{r['accuracy']:>10.4f} {r['macro_f1']:>10.4f}{marker}")

save = {n: {k: v for k, v in r.items() if k not in ["preds","labels"]}
        for n, r in all_results.items()}
with open(OUTPUT_DIR / "all_results_18cls.json", "w") as f:
    json.dump(save, f, indent=2)
print("\nresults saved to all_results_18cls.json")


## master comparison chart

In [ ]:
names  = [n for n, _ in sorted_res]
accs   = [r["accuracy"] for _, r in sorted_res]
f1s    = [r["macro_f1"] for _, r in sorted_res]
types  = [r["model_type"] for _, r in sorted_res]
colors = {"traditional_ml": "#2c7bb6", "deep_learning": "#1a9641", "hierarchical": "#d7191c"}
bcols  = [colors.get(t, "#888") for t in types]

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle("model comparison -- 18-class e-waste (8 HIGH / 4 MEDIUM / 6 LOW hazard)",
             fontsize=13, fontweight="bold")

x = np.arange(len(names))
for ax, vals, xlabel, title in zip(
    axes, [accs, f1s], ["accuracy", "macro f1"], ["test accuracy", "macro f1 score"]):
    bars = ax.barh(x, vals, color=bcols, alpha=0.85, edgecolor="white", height=0.6)
    ax.axvline(x=0.95, color="red", linestyle="--", alpha=0.7, linewidth=1.5,
               label="target = 0.95")
    for bar, val in zip(bars, vals):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f"{val:.4f}", va="center", fontsize=8)
    ax.set_yticks(x)
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.set_xlim(0, 1.1)
    ax.legend(fontsize=9)
    ax.grid(axis="x", alpha=0.3)

patches = [mpatches.Patch(color=c, label=t) for t, c in colors.items()]
fig.legend(handles=patches, loc="lower center", ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.savefig(GRAPHS_DIR / "master_comparison_18cls.png", dpi=150, bbox_inches="tight")
plt.close()
print("master comparison chart saved")


## hazard-level routing accuracy -- sdg 12.4

In [ ]:
print("hazard-level routing accuracy -- sdg 12.4 disposal compliance")
print("=" * 68)
print(f"{'model':<28} {'hazard_acc':>12} {'hazard_f1':>12}")
print("-" * 55)

HINT = {"HIGH": 0, "MEDIUM": 1, "LOW": 2}
hazard_report = {}

for name, r in sorted_res:
    ph = [HINT[HAZARD_MAP[CLASS_NAMES[p]]] for p in r["preds"]]
    th = [HINT[HAZARD_MAP[CLASS_NAMES[l]]] for l in r["labels"]]
    ha = accuracy_score(th, ph)
    hf = f1_score(th, ph, average="macro")
    hazard_report[name] = {"hazard_accuracy": round(ha, 4), "hazard_f1": round(hf, 4)}
    print(f"  {name:<26} {ha:>12.4f} {hf:>12.4f}")

with open(OUTPUT_DIR / "hazard_routing_18cls.json", "w") as f:
    json.dump(hazard_report, f, indent=2)
print("\nhazard routing results saved")


## disposal database -- sdg 12.4 reference

In [ ]:
disposal_db = {}
for cls in CLASS_NAMES:
    disposal_db[cls] = {
        "hazard_level"        : HAZARD_MAP[cls],
        "material_composition": MATERIAL_MAP[cls],
        "disposal_pathway"    : DISPOSAL_MAP[cls],
        "sdg_target"          : "SDG 12.4" if HAZARD_MAP[cls] in ["HIGH", "MEDIUM"] else "SDG 12.5",
        "unu_keys_category"   : (
            "temperature exchange equipment" if cls in ["Refrigerator", "Air-Conditioner"]
            else "screens and monitors"       if cls in ["Television", "Laptop"]
            else "lamps"                      if cls == "light bulbs"
            else "large equipment"            if cls in ["Washing Machine", "Microwave", "Printer"]
            else "small IT and telecom"       if cls in ["Mobile", "Battery", "PCB", "Keyboard", "Mouse", "Microchip-IC"]
            else "small equipment components"
        )
    }

with open(OUTPUT_DIR / "disposal_database_18cls.json", "w") as f:
    json.dump(disposal_db, f, indent=2)

print("sdg 12.4 disposal database (18 classes)")
print("=" * 90)
print(f"{'class':<25} {'hazard':<10} {'sdg':<10} {'unu-keys category'}")
print("-" * 90)
for cls, info in disposal_db.items():
    print(f"  {cls:<23} {info['hazard_level']:<10} {info['sdg_target']:<10} {info['unu_keys_category']}")
print("\ndisposal database saved")


## per-class f1 heatmap -- all models

In [ ]:
model_names = [n for n, _ in sorted_res]
f1_matrix   = np.zeros((len(model_names), NUM_CLASSES))

for i, (name, r) in enumerate(sorted_res):
    f1_matrix[i] = f1_score(r["labels"], r["preds"],
                             average=None, labels=list(range(NUM_CLASSES)))

fig, ax = plt.subplots(figsize=(26, 12))
sns.heatmap(f1_matrix, ax=ax,
            xticklabels=CLASS_NAMES, yticklabels=model_names,
            annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=0.5, vmax=1.0, linewidths=0.3)
ax.set_title("per-class f1 score -- all models vs all 18 classes",
             fontsize=13, fontweight="bold")
ax.tick_params(axis="x", rotation=45, labelsize=9)
ax.tick_params(axis="y", rotation=0, labelsize=9)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / "per_class_f1_heatmap_18cls.png", dpi=150, bbox_inches="tight")
plt.close()
print("per-class f1 heatmap saved")
